# 18 Build OTM Base

Turn the expanded OpenTripMap collection into a cleaned project POI base.

In [17]:
import ast
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [18]:
RADIUS_PATH = "../data/processed/opentripmap_expanded_radius_results.csv"
DETAILS_PATH = "../data/processed/opentripmap_expanded_details_results.csv"

OUTPUT_BASE_PATH = "../data/processed/otm_pois_base.csv"
OUTPUT_REVIEW_PATH = "../data/processed/otm_pois_base_review.csv"

radius_df = pd.read_csv(RADIUS_PATH)
details_df = pd.read_csv(DETAILS_PATH)

print("radius_df:", radius_df.shape)
print("details_df:", details_df.shape)


radius_df: (395, 9)
details_df: (20, 17)


## Helpers

In [19]:
def parse_dict_like(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    text = str(value).strip()
    if not text:
        return {}
    try:
        return ast.literal_eval(text)
    except Exception:
        return {}


def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.casefold()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def build_display_name(name):
    return "" if pd.isna(name) else str(name).strip()


def map_category_clean(kinds):
    kinds = "" if pd.isna(kinds) else str(kinds).casefold()
    if "museum" in kinds or "museums" in kinds:
        return "museum"
    if "palaces" in kinds or "historic_architecture" in kinds or "monument" in kinds or "monuments" in kinds or "historic" in kinds or "archaeology" in kinds or "fortifications" in kinds or "aqueducts" in kinds:
        return "historic"
    if "mosques" in kinds or "churches" in kinds or "synagogues" in kinds or "cathedrals" in kinds or "religion" in kinds or "other_temples" in kinds:
        return "religious"
    return "attraction"


NOISE_PATTERNS = [
    r"\bhistoric areas of istanbul\b",
    r"\brunic inscriptions?\b",
    r"\bnizam pide\b",
    r"\b^pera$\b",
    r"\bpasha$",
]

NOISE_KINDS = [
    "restaurants",
    "foods",
    "bars",
    "nightclubs",
    "adult",
    "nature_reserves",
    "other_nature_conservation_areas",
]


def is_obvious_noise(name, kinds):
    norm_name = normalize_text(name)
    kinds_text = "" if pd.isna(kinds) else str(kinds).casefold()

    if any(re.search(pattern, norm_name) for pattern in NOISE_PATTERNS):
        return True

    if any(token in kinds_text for token in NOISE_KINDS):
        if "museum" not in kinds_text and "historic" not in kinds_text and "palaces" not in kinds_text:
            return True

    return False


## Parse and merge

In [20]:
point_parsed = radius_df["point"].apply(parse_dict_like)
radius_df["lat"] = point_parsed.apply(lambda p: p.get("lat"))
radius_df["lon"] = point_parsed.apply(lambda p: p.get("lon"))

address_parsed = details_df["address"].apply(parse_dict_like)
details_df["country_code"] = address_parsed.apply(lambda p: p.get("country_code"))
details_df["town"] = address_parsed.apply(lambda p: p.get("town"))
details_df["suburb"] = address_parsed.apply(lambda p: p.get("suburb"))

if "preview" in details_df.columns:
    preview_parsed = details_df["preview"].apply(parse_dict_like)
    details_df["preview_source"] = preview_parsed.apply(lambda p: p.get("source"))

detail_keep_cols = [c for c in [
    "xid", "wikipedia", "wikidata", "url", "address", "preview_source", "town", "suburb", "country_code", "wikipedia_extracts"
] if c in details_df.columns]

otm_df = radius_df.merge(details_df[detail_keep_cols], on="xid", how="left", suffixes=("", "_detail"))

detail_wikidata_col = "wikidata_detail"
if detail_wikidata_col in otm_df.columns:
    otm_df["wikidata"] = otm_df[detail_wikidata_col].combine_first(otm_df["wikidata"])

otm_df["display_name_en"] = otm_df["name"].apply(build_display_name)
otm_df["category_clean"] = otm_df["kinds"].apply(map_category_clean)
otm_df["poi_id"] = otm_df["xid"].apply(lambda x: f"otm_{x}")
otm_df["source"] = "opentripmap"
otm_df["dist_from_query_center_m"] = otm_df["dist"]
otm_df["wikipedia_url"] = otm_df["wikipedia"] if "wikipedia" in otm_df.columns else pd.NA
otm_df["address_raw"] = otm_df["address"] if "address" in otm_df.columns else pd.NA

print("Merged shape:", otm_df.shape)
otm_df[["xid", "name", "display_name_en", "category_clean", "wikidata", "wikipedia_url"]].head(20)


Merged shape: (395, 27)


,xid,name,display_name_en,category_clean,wikidata,wikipedia_url
0,N7215645385,The Blue Mosque,The Blue Mosque,religious,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...
1,R1564032,Süleymaniye Mosque,Süleymaniye Mosque,religious,Q178643,https://en.wikipedia.org/wiki/S%C3%BCleymaniye...
2,Q5773394,Historic Areas of Istanbul,Historic Areas of Istanbul,attraction,Q5773394,https://en.wikipedia.org/wiki/Historic%20Areas...
3,R1555271,Hagia Sophia Grand Mosque,Hagia Sophia Grand Mosque,museum,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia
4,W109862851,The Hagia Sophia Grand Mosque,The Hagia Sophia Grand Mosque,museum,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia
5,Q2656937,Runic inscriptions in Hagia Sophia,Runic inscriptions in Hagia Sophia,historic,Q2656937,https://en.wikipedia.org/wiki/Runic%20inscript...
6,N415157636,Serpent Column,Serpent Column,historic,Q588892,https://en.wikipedia.org/wiki/Serpent%20Column
7,W103953125,Tomb of Sultan Ahmet,Tomb of Sultan Ahmet,religious,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...
8,W326372295,Yıldız Palace,Yıldız Palace,museum,Q911734,https://en.wikipedia.org/wiki/Y%C4%B1ld%C4%B1z...
9,N4477143891,Yıldız Palace,Yıldız Palace,historic,Q911734,https://en.wikipedia.org/wiki/Y%C4%B1ld%C4%B1z...


## Remove obvious junk

In [21]:
otm_df["is_obvious_noise"] = otm_df.apply(lambda row: is_obvious_noise(row["display_name_en"], row["kinds"]), axis=1)

print("Noise rows:", int(otm_df["is_obvious_noise"].sum()))
display(otm_df.loc[otm_df["is_obvious_noise"], ["xid", "name", "query_area", "kinds"]].head(40))

otm_df = otm_df.loc[~otm_df["is_obvious_noise"]].copy()
print("After noise filter:", otm_df.shape)


Noise rows: 8


,xid,name,query_area,kinds
2,Q5773394,Historic Areas of Istanbul,Hagia Sophia / Basilica Cistern,"interesting_places,natural,nature_reserves,oth..."
5,Q2656937,Runic inscriptions in Hagia Sophia,Hagia Sophia / Basilica Cistern,"other,unclassified_objects,interesting_places,..."
43,N8010273385,Nizam Pide Galatasaray,Istiklal / Pera,"cultural,museums,interesting_places,restaurant..."
78,N6185582262,pera,Istiklal / Pera,"cultural,museums,interesting_places,restaurant..."
129,N2771801929,Pasha,Sultanahmet Core,"palaces,architecture,historic_architecture,int..."
208,Q28098273,Reina,Ortakoy,"cultural,theatres_and_entertainments,nightclub..."
357,Q6060754,Naile Sultan Grove,Ortakoy,"interesting_places,natural,nature_reserves,oth..."
384,N5243950801,Hirka-i sherif,Balat / Fener,"religion,mosques,interesting_places,restaurant..."


After noise filter: (387, 28)


## Deduplicate and flag review cases

In [22]:
otm_df["norm_name"] = otm_df["display_name_en"].apply(normalize_text)
otm_df["lat_round"] = otm_df["lat"].round(4)
otm_df["lon_round"] = otm_df["lon"].round(4)

otm_df["dedupe_key"] = otm_df.apply(
    lambda row: row["wikidata"]
    if pd.notna(row["wikidata"]) and str(row["wikidata"]).strip()
    else row["wikipedia_url"]
    if pd.notna(row["wikipedia_url"]) and str(row["wikipedia_url"]).strip()
    else f"{row['norm_name']}|{row['lat_round']}|{row['lon_round']}",
    axis=1,
)

otm_df["candidate_count_same_key"] = otm_df.groupby("dedupe_key")["xid"].transform("count")
otm_df["needs_review"] = 0
otm_df.loc[otm_df["candidate_count_same_key"] > 1, "needs_review"] = 1
otm_df.loc[otm_df["wikipedia_url"].isna() & otm_df["wikidata"].isna(), "needs_review"] = 1
otm_df.loc[otm_df["display_name_en"].str.len().fillna(0) < 3, "needs_review"] = 1

review_df = otm_df.loc[otm_df["candidate_count_same_key"] > 1].copy()
print("Review rows:", review_df.shape)
review_df[["xid", "name", "query_area", "wikidata", "wikipedia_url", "candidate_count_same_key"]].head(50)


Review rows: (104, 34)


,xid,name,query_area,wikidata,wikipedia_url,candidate_count_same_key
0,N7215645385,The Blue Mosque,Sultanahmet Core,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...,2
3,R1555271,Hagia Sophia Grand Mosque,Hagia Sophia / Basilica Cistern,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia,2
4,W109862851,The Hagia Sophia Grand Mosque,Hagia Sophia / Basilica Cistern,Q12506,https://en.wikipedia.org/wiki/Hagia%20Sophia,2
7,W103953125,Tomb of Sultan Ahmet,Sultanahmet Core,Q80541,https://en.wikipedia.org/wiki/Sultan%20Ahmed%2...,2
8,W326372295,Yıldız Palace,Yildiz Palace,Q911734,https://en.wikipedia.org/wiki/Y%C4%B1ld%C4%B1z...,2
9,N4477143891,Yıldız Palace,Yildiz Palace,Q911734,https://en.wikipedia.org/wiki/Y%C4%B1ld%C4%B1z...,2
10,N7294561685,Chora Mosque,Balat / Fener,Q849489,https://en.wikipedia.org/wiki/Chora%20Church,2
11,W32395058,Kariye Museum,Balat / Fener,Q849489,https://en.wikipedia.org/wiki/Chora%20Church,2
12,R7318154,Rumeli Fortress,Rumeli Hisari,Q90801,https://en.wikipedia.org/wiki/Rumelihisar%C4%B1,2
13,W102190099,Osman ağa Mosque,Kadikoy Historic Center,Q6029146,https://tr.wikipedia.org/wiki/Osmana%C4%9Fa%20...,2


## Build base table

In [23]:
base_df = (
    otm_df.sort_values(
        ["candidate_count_same_key", "rate", "dist_from_query_center_m"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset=["dedupe_key"], keep="first")
    .copy()
)

base_df = base_df[[
    "poi_id",
    "xid",
    "name",
    "display_name_en",
    "kinds",
    "category_clean",
    "wikidata",
    "wikipedia_url",
    "lat",
    "lon",
    "query_area",
    "dist_from_query_center_m",
    "rate",
    "address_raw",
    "preview_source",
    "source",
    "needs_review",
    "candidate_count_same_key",
]]

base_df = base_df.rename(columns={
    "xid": "otm_xid",
    "preview_source": "preview_image",
})

print("Base shape:", base_df.shape)
print(base_df["category_clean"].value_counts(dropna=False).to_string())
print("with wikidata:", int(base_df["wikidata"].notna().sum()))
print("with wikipedia_url:", int(base_df["wikipedia_url"].notna().sum()))

base_df.head(30)


Base shape: (327, 18)
category_clean
religious     139
historic       95
museum         49
attraction     44
with wikidata: 326
with wikipedia_url: 13


,poi_id,otm_xid,name,display_name_en,kinds,category_clean,wikidata,wikipedia_url,lat,lon,query_area,dist_from_query_center_m,rate,address_raw,preview_image,source,needs_review,candidate_count_same_key
1,otm_R1564032,R1564032,Süleymaniye Mosque,Süleymaniye Mosque,"religion,mosques,interesting_places",religious,Q178643,https://en.wikipedia.org/wiki/S%C3%BCleymaniye...,41.016300,28.963978,Suleymaniye,18.644092,7,"{'city': 'Süleymaniye Mahallesi', 'road': 'Pro...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
6,otm_N415157636,N415157636,Serpent Column,Serpent Column,"historic,monuments_and_memorials,burial_places...",historic,Q588892,https://en.wikipedia.org/wiki/Serpent%20Column,41.005661,28.975103,Sultanahmet Core,145.719100,7,"{'city': 'Binbirdirek Mahallesi', 'town': 'Fat...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
14,otm_W23236783,W23236783,Galata Tower,Galata Tower,"architecture,towers,interesting_places,observa...",attraction,Q91274,https://en.wikipedia.org/wiki/Galata%20Tower,41.025642,28.974213,Galata Tower,17.032431,3,"{'city': 'Bereketzade Mahallesi', 'town': 'Bey...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
16,otm_Q105840441,Q105840441,Mısırlı Osman Ağa Fountain,Mısırlı Osman Ağa Fountain,"fountains,historic,cultural,urban_environment,...",historic,Q105840441,https://tr.wikipedia.org/wiki/M%C4%B1s%C4%B1rl...,40.991074,29.026735,Kadikoy Historic Center,41.479630,3,"{'city': 'Osmanağa Mahallesi', 'road': 'Söğütl...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
17,otm_N4519290891,N4519290891,Selman aga Camii,Selman aga Camii,"religion,mosques,interesting_places",religious,Q6056669,https://tr.wikipedia.org/wiki/Selman%20A%C4%9F...,41.025700,29.015795,Uskudar Waterfront,51.282068,3,"{'road': 'Selmani Pak Caddesi', 'town': 'Üsküd...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
18,otm_W261825033,W261825033,Church of St. Mary of the Mongols,Church of St. Mary of the Mongols,"religion,churches,interesting_places,eastern_o...",religious,Q1950180,https://en.wikipedia.org/wiki/Church%20of%20Sa...,41.029602,28.949026,Balat / Fener,57.655526,3,"{'road': 'Tevki Cafer Mektebi Sokağı', 'town':...",https://upload.wikimedia.org/wikipedia/commons...,opentripmap,0,1
20,otm_Q3082387,Q3082387,Church of the Virgin of the Pharos,Church of the Virgin of the Pharos,"religion,churches,interesting_places,other_chu...",religious,Q3082387,NaN,41.005833,28.977222,Sultanahmet Core,59.812504,3,NaN,NaN,opentripmap,0,1
25,otm_N4591192493,N4591192493,Spice Bazaar,Spice Bazaar,"view_points,other,interesting_places,shops,mar...",attraction,Q668641,NaN,41.016281,28.971256,Eminonu / Spice Bazaar,92.140158,3,NaN,NaN,opentripmap,0,1
26,otm_N3386270532,N3386270532,Galatasaray Museum,Galatasaray Museum,"cultural,museums,interesting_places,art_galleries",museum,Q3329724,NaN,41.033482,28.976940,Istiklal / Pera,93.163832,3,NaN,NaN,opentripmap,0,1
29,otm_Q16947921,Q16947921,Fountain of Ahmed III (Üsküdar),Fountain of Ahmed III (Üsküdar),"fountains,historic,cultural,urban_environment,...",historic,Q16947921,NaN,41.026787,29.015354,Uskudar Waterfront,110.383145,3,NaN,NaN,opentripmap,0,1


## Save outputs

In [24]:
base_df.to_csv(OUTPUT_BASE_PATH, index=False)
review_df.to_csv(OUTPUT_REVIEW_PATH, index=False)

print("Saved:", OUTPUT_BASE_PATH, base_df.shape)
print("Saved:", OUTPUT_REVIEW_PATH, review_df.shape)


Saved: ../data/processed/otm_pois_base.csv (327, 18)
Saved: ../data/processed/otm_pois_base_review.csv (104, 34)
